**Quick setup**: Install dependencies if needed (run in the notebook or environment).

Run this once in your environment:

In [1]:
# If running in a fresh environment, install required packages (uncomment to run).
# Note: prefer using your conda env from `environment_selfies.yml` for RDKit if needed.
# !pip install -q transformers datasets evaluate scikit-learn accelerate

In [2]:
# Imports (restricted to GPU 1)
import os
# Force notebook to see only GPU 1 (the second physical GPU). RESTART kernel after setting if torch was previously imported.
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('libs imported')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Visible GPUs: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'Visible GPU {i}: {torch.cuda.get_device_name(i)}')

/home/barradd/Documents/GitHub/machine_learning_chem_RGS/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


libs imported
CUDA available: True
Visible GPUs: 1
Visible GPU 0: Quadro GV100


In [ ]:
os.environ['TOKENIZERS_PARALLELISM'] = 'true'

# Load the larger dataset built from `dv_values_by_cid.csv`
data_files = {
    'train': '../data/hf_dataset_large_train.csv',
    'validation': '../data/hf_dataset_large_val.csv',
    'test': '../data/hf_dataset_large_test.csv',
}
dataset = load_dataset('csv', data_files=data_files)
dataset

Generating train split: 155218 examples [00:00, 524726.63 examples/s]
Generating train split: 155218 examples [00:00, 524726.63 examples/s]
Generating validation split: 33261 examples [00:00, 627137.54 examples/s]
Generating validation split: 33261 examples [00:00, 627137.54 examples/s]
Generating test split: 33262 examples [00:00, 617731.44 examples/s]



DatasetDict({
    train: Dataset({
        features: ['text', 'DV_C', 'DV_n'],
        num_rows: 155218
    })
    validation: Dataset({
        features: ['text', 'DV_C', 'DV_n'],
        num_rows: 33261
    })
    test: Dataset({
        features: ['text', 'DV_C', 'DV_n'],
        num_rows: 33262
    })
})

In [4]:
# Inspect a few examples
for split in dataset:
    print(split, len(dataset[split]))
    print(dataset[split][0])


train 155218
{'text': '[C][=C][C][=C][C][=Branch1][Branch1][=C][N][Ring1][Branch1][C][=C][C][=C][Branch1][Branch1][C][=C][Ring1][=Branch1][C][=O]', 'DV_C': 8.621359889999548, 'DV_n': 9.3}
validation 33261
{'text': '[C][C][Branch1][Branch1][C][O][Ring1][Ring2][Branch1][=Branch2][C][=C][C][=C][C][=C][Ring1][=Branch1][N]', 'DV_C': 3.852283889999516, 'DV_n': 2.5}
test 33262
{'text': '[C][C].[C][C][=C][C][=C][Ring1][Branch1][C][=C][C][=C][C][=C][C][=C][Ring1][=Branch1][Ring1][#Branch2]', 'DV_C': -0.540286110000279, 'DV_n': -1.2}


## Tokenization
We use a standard tokenizer (`distilbert-base-uncased`) for this PoC. For SELFIES-specific modeling, replace with a tokenizer trained for SELFIES tokens or a character-level tokenizer consistent with your pretrained model.

In [5]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding=False)

tokenized = dataset.map(tokenize_fn, batched=True)
# Remove the raw text column and rename 'DV_n' to 'labels' for Trainer
tokenized = tokenized.remove_columns(['text','DV_C'])
tokenized = tokenized.rename_column('DV_n', 'labels')

# Ensure labels are float32 (not float64) and properly shaped for regression
def fix_labels(batch):
    batch['labels'] = np.array(batch['labels'], dtype=np.float32)
    return batch

tokenized = tokenized.map(fix_labels, batched=True)
tokenized

Map: 100%|██████████| 33262/33262 [00:00<00:00, 337241.99 examples/s]


DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 155218
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 33261
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 33262
    })
})

## Model (regression)
We load a pretrained encoder and attach a regression head (num_labels=1).

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
# Ensure the model is configured for regression
model.config.problem_type = 'regression'

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer)

print('model and data collator ready')

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model and data collator ready


In [7]:
# Compute metrics for regression
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    # Flatten predictions to 1D
    preds = np.squeeze(preds)
    # Ensure labels are also 1D
    labels = np.squeeze(labels)
    mse = mean_squared_error(labels, preds)
    return {
        'mse': float(mse),
        'rmse': float(np.sqrt(mse)),
        'mae': float(mean_absolute_error(labels, preds)),
        'r2': float(r2_score(labels, preds)),
    }

# Single GPU training args (GPU 1 only is visible due to CUDA_VISIBLE_DEVICES)
training_args = TrainingArguments(
    output_dir='../data/results_poc',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=10,
    learning_rate=5e-5,
    remove_unused_columns=False,
    fp16=True,                    # Mixed precision on V100
    gradient_checkpointing=True,  # Memory saving
    dataloader_num_workers=2,
    optim='adamw_torch',
    report_to=[],
    label_names=['labels'],       # Explicitly specify label column
)

# Optional performance tweaks
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('trainer ready (single GPU mode)')
print(f'Visible GPU count: {torch.cuda.device_count()}')

trainer ready (single GPU mode)
Visible GPU count: 1


In [8]:
# Diagnostic: verify process bound to the intended GPU only
if torch.cuda.is_available():
    current = torch.cuda.current_device()
    print(f'Current CUDA device index (should be 0 now): {current}')
    print(f'Device name: {torch.cuda.get_device_name(current)}')
else:
    print('CUDA not available after restriction.')

Current CUDA device index (should be 0 now): 0
Device name: Quadro GV100


## Train (PoC)
Run the training loop. With the tiny dataset this will quickly overfit — expect meaningless metrics, but it verifies the pipeline.

In [9]:
# Robust training wrapper with diagnostics
from contextlib import suppress

def print_gpu_mem(prefix=""):
    if torch.cuda.is_available():
        stats = []
        for i in range(torch.cuda.device_count()):
            allocated = torch.cuda.memory_allocated(i) / 1024**2
            reserved = torch.cuda.memory_reserved(i) / 1024**2
            stats.append(f"GPU{i} alloc={allocated:.1f}MB reserved={reserved:.1f}MB")
        print(prefix + " | ".join(stats))

print_gpu_mem("Before training")
try:
    train_results = trainer.train()
    print_gpu_mem("After training")
    print(train_results)
except RuntimeError as e:
    # Common CUDA OOM or AMP errors
    if 'CUDA out of memory' in str(e):
        print('CUDA OOM detected. Suggest lowering batch size further (e.g. 2) or disabling fp16.')
    elif 'cublas' in str(e).lower():
        print('cuBLAS error. Potential driver/library mismatch. Consider reinstalling matching torch + CUDA.')
    else:
        print('RuntimeError during training:', e)
    # Cleanup
    if torch.cuda.is_available():
        with suppress(Exception):
            torch.cuda.empty_cache()
        print_gpu_mem("After cleanup")

Before trainingGPU0 alloc=256.5MB reserved=292.0MB


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch,Training Loss,Validation Loss,Mse,Rmse,Mae,R2
1,49.164800,44.212471,44.212471,6.649246,5.441464,0.102298
2,30.140000,26.479887,26.479885,5.145861,3.934924,0.462345
3,21.294000,20.857279,20.857279,4.566977,3.295549,0.576508


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

After trainingGPU0 alloc=788.2MB reserved=1412.0MB
TrainOutput(global_step=116415, training_loss=35.02696444837415, metrics={'train_runtime': 7073.7048, 'train_samples_per_second': 65.829, 'train_steps_per_second': 16.457, 'total_flos': 1.1059072781109444e+16, 'train_loss': 35.02696444837415, 'epoch': 3.0})


In [10]:
# Evaluate on the test split
metrics = trainer.evaluate(eval_dataset=tokenized['test'])
print('Test metrics:', metrics)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Test metrics: {'eval_loss': 20.753541946411133, 'eval_mse': 20.753541946411133, 'eval_rmse': 4.555605552109525, 'eval_mae': 3.289539098739624, 'eval_r2': 0.5812366008758545, 'eval_runtime': 52.3665, 'eval_samples_per_second': 635.177, 'eval_steps_per_second': 158.804, 'epoch': 3.0}


## Save the model
Save the finetuned model locally for quick inference tests.

In [11]:
os.makedirs('../models/poc_distilbert_regressor', exist_ok=True)
trainer.save_model('../models/poc_distilbert_regressor')
print('model saved to ../models/poc_distilbert_regressor')

model saved to ../models/poc_distilbert_regressor


In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

: 

## Next steps / Notes
- Replace `distilbert-base-uncased` with a SELFIES-aware model/tokenizer if available (recommended).
- Increase dataset size (convert `data/dv_values_by_cid.csv` SMILES → SELFIES) before meaningful training.
- Use `transformers` training optimizations (FP16, gradient accumulation, `accelerate`) for larger runs.